# Deep Learning Classification of Bovine Liver Lesions from Smartphone Images
### A Ghanaian Abattoir Study — Full Analysis Notebook

This notebook runs the complete analysis pipeline described in the proposal:

1. **Setup** — install/import dependencies
2. **Data loading & descriptive statistics** — dataset summary, lesion frequency
3. **Dataset & augmentation** — PyTorch `Dataset` + transforms
4. **Deep learning models** — transfer learning (MobileNetV2 / EfficientNet-B0 / ResNet18)
5. **Stratified k-fold cross-validation** — training + evaluation loop
6. **Results** — metrics table, ROC curves, confusion matrices
7. **Classical ML baseline** — colour/texture features + SVM/Random Forest, same CV protocol
8. **Deep learning vs. classical comparison**
9. **Grad-CAM** — interpretability visualisation on the best model
10. **Summary & next steps**

**Before running:** place your images in this structure, next to the notebook:

```
data/
  negative/   (60 images — normal liver)
  positive/   (40 images — visible gross lesion)
```

Run cells top to bottom. Cells are self-contained per section so you can re-run
a section (e.g. just the classical baseline) without re-running everything.


## 1. Setup

In [ ]:
# Run once. Comment out after first successful install to save time on re-runs.
# !pip install torch torchvision scikit-learn scikit-image pandas numpy matplotlib pillow opencv-python-headless --quiet


In [ ]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models

from sklearn.model_selection import StratifiedKFold, LeaveOneOut
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, roc_curve,
    precision_score, f1_score,
)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


In [ ]:
# ---- Configuration ----
DATA_DIR = Path("data")          # expects DATA_DIR/negative and DATA_DIR/positive
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

IMAGE_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 20
LR = 1e-4
N_FOLDS = 5              # set to -1 for leave-one-out if your sample is very small
BACKBONE = "mobilenet_v2"  # "mobilenet_v2" | "efficientnet_b0" | "resnet18"
FREEZE_BACKBONE = False    # True = train only the classifier head (often more stable at small n)

print(f"Data dir: {DATA_DIR.resolve()}")


## 2. Data Loading & Descriptive Statistics

This section plays the role of the Excel/SPSS descriptive-statistics step in the
proposal (§2.8): summarising class counts, image properties, and (if you've
recorded lesion sub-type metadata) lesion-type frequencies.


In [ ]:
def scan_dataset(data_dir):
    records = []
    for label, cls in enumerate(["negative", "positive"]):
        cls_dir = data_dir / cls
        if not cls_dir.exists():
            raise FileNotFoundError(f"Expected folder {cls_dir} not found.")
        for p in sorted(cls_dir.iterdir()):
            if p.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                with Image.open(p) as img:
                    w, h = img.size
                records.append({
                    "path": str(p), "class": cls, "label": label,
                    "width": w, "height": h,
                })
    return pd.DataFrame(records)

df_images = scan_dataset(DATA_DIR)
print(f"Total images: {len(df_images)}")
df_images["class"].value_counts()


In [ ]:
# Basic descriptive stats: class balance + image dimension summary
display_summary = df_images.groupby("class")[["width", "height"]].describe()
display_summary


In [ ]:
# Class balance bar chart
counts = df_images["class"].value_counts()
plt.figure(figsize=(4, 4))
plt.bar(counts.index, counts.values, color=["#4C72B0", "#C44E52"])
plt.ylabel("Number of images")
plt.title("Class balance")
for i, v in enumerate(counts.values):
    plt.text(i, v + 0.5, str(v), ha="center")
plt.tight_layout()
plt.savefig(OUT_DIR / "class_balance.png", dpi=150)
plt.show()


In [ ]:
# OPTIONAL: if you have a metadata CSV with lesion sub-type labels
# (fascioliasis / abscess / fibrosis / cirrhosis / fatty liver / normal),
# load it here to get lesion-type frequency counts as described in the proposal.
#
# metadata_path = DATA_DIR / "lesion_metadata.csv"
# if metadata_path.exists():
#     df_meta = pd.read_csv(metadata_path)
#     lesion_counts = df_meta["lesion_type"].value_counts()
#     lesion_counts.plot(kind="bar", figsize=(6,4), title="Lesion type frequency")
#     plt.tight_layout()
#     plt.savefig(OUT_DIR / "lesion_type_frequency.png", dpi=150)
#     plt.show()
# else:
#     print("No lesion_metadata.csv found — skip if only binary labels collected so far.")


## 3. PyTorch Dataset & Augmentation

In [ ]:
class LesionDataset(Dataset):
    """Expects data_dir/negative/*.jpg and data_dir/positive/*.jpg"""

    def __init__(self, data_dir, transform=None):
        self.data_dir = Path(data_dir)
        self.transform = transform
        self.samples = []
        for label, cls in enumerate(["negative", "positive"]):
            cls_dir = self.data_dir / cls
            for img_path in sorted(cls_dir.iterdir()):
                if img_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                    self.samples.append((img_path, label))
        if len(self.samples) == 0:
            raise RuntimeError(f"No images found under {self.data_dir}")
        self.labels = np.array([s[1] for s in self.samples])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


def get_transforms(image_size=IMAGE_SIZE):
    train_tf = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(25),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.02),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
    ])
    val_tf = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return train_tf, val_tf

train_tf, val_tf = get_transforms()
print("Transforms ready.")


## 4. Deep Learning Model Builder (Transfer Learning)

In [ ]:
def build_model(backbone_name, freeze_backbone=False):
    if backbone_name == "mobilenet_v2":
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, 1)
        backbone_params = model.features.parameters()
    elif backbone_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, 1)
        backbone_params = model.features.parameters()
    elif backbone_name == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, 1)
        backbone_params = [p for n, p in model.named_parameters() if not n.startswith("fc")]
    else:
        raise ValueError(f"Unknown backbone {backbone_name}")

    if freeze_backbone:
        for p in backbone_params:
            p.requires_grad = False
    return model

# Quick smoke test
_m = build_model(BACKBONE, FREEZE_BACKBONE)
n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"{BACKBONE}: {n_params:,} trainable parameters")
del _m


## 5. Stratified K-Fold Cross-Validation — Training & Evaluation

With only 100 images, a single train/test split gives a noisy performance
estimate. Stratified k-fold cross-validation is used instead, preserving the
class ratio in every fold. Class-weighted loss handles the 60/40 imbalance.


In [ ]:
def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    specificity = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
    precision = precision_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = float("nan")
    return {
        "sensitivity_recall": sensitivity, "specificity": specificity,
        "precision": precision, "f1": f1, "auc": auc,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def run_fold(fold_idx, train_ds, val_ds, epochs=EPOCHS, lr=LR,
             backbone=BACKBONE, freeze_backbone=FREEZE_BACKBONE, batch_size=BATCH_SIZE):
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    model = build_model(backbone, freeze_backbone).to(DEVICE)

    train_labels = np.array([train_ds.dataset.labels[i] for i in train_ds.indices])
    n_neg, n_pos = (train_labels == 0).sum(), (train_labels == 1).sum()
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable_params, lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_auc, best_state = -1, None

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.float().unsqueeze(1).to(DEVICE)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
        scheduler.step()
        train_loss = running_loss / len(train_loader.dataset)

        model.eval()
        val_probs, val_labels = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                outputs = model(imgs.to(DEVICE))
                probs = torch.sigmoid(outputs).cpu().numpy().flatten()
                val_probs.extend(probs)
                val_labels.extend(labels.numpy())
        val_labels_arr, val_probs_arr = np.array(val_labels), np.array(val_probs)
        try:
            val_auc = roc_auc_score(val_labels_arr, val_probs_arr)
        except ValueError:
            val_auc = float("nan")

        if (epoch + 1) % 5 == 0 or epoch == epochs - 1:
            print(f"  [fold {fold_idx}] epoch {epoch+1}/{epochs}  "
                  f"train_loss={train_loss:.4f}  val_auc={val_auc:.4f}")

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.eval()
    val_probs, val_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            outputs = model(imgs.to(DEVICE))
            probs = torch.sigmoid(outputs).cpu().numpy().flatten()
            val_probs.extend(probs)
            val_labels.extend(labels.numpy())

    return np.array(val_labels), np.array(val_probs), model

print("Training functions ready.")


In [ ]:
# ---- Run the full cross-validation loop ----
base_dataset = LesionDataset(DATA_DIR, transform=None)
train_dataset_aug = LesionDataset(DATA_DIR, transform=train_tf)
val_dataset_plain = LesionDataset(DATA_DIR, transform=val_tf)

labels = base_dataset.labels
print(f"Dataset: {len(labels)} images ({(labels==0).sum()} negative, {(labels==1).sum()} positive)")

if N_FOLDS == -1:
    splitter = LeaveOneOut()
    split_iter = splitter.split(np.zeros(len(labels)))
    n_splits = len(labels)
else:
    splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    split_iter = splitter.split(np.zeros(len(labels)), labels)
    n_splits = N_FOLDS

dl_fold_metrics = []
dl_roc_data = []
dl_cm_data = []

for fold_idx, (train_idx, val_idx) in enumerate(split_iter, start=1):
    print(f"\n=== DL Fold {fold_idx}/{n_splits} (train={len(train_idx)}, val={len(val_idx)}) ===")
    train_subset = Subset(train_dataset_aug, train_idx)
    val_subset = Subset(val_dataset_plain, val_idx)

    y_true, y_prob, model = run_fold(fold_idx, train_subset, val_subset)
    metrics = compute_metrics(y_true, y_prob)
    metrics["fold"] = fold_idx
    dl_fold_metrics.append(metrics)
    print(f"  Fold {fold_idx} metrics: {metrics}")

    torch.save(model.state_dict(), OUT_DIR / f"best_model_fold{fold_idx}.pt")

    if not np.isnan(metrics["auc"]):
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        dl_roc_data.append((fold_idx, fpr, tpr, metrics["auc"]))
    dl_cm_data.append((fold_idx, y_true, (y_prob >= 0.5).astype(int)))

df_dl_metrics = pd.DataFrame(dl_fold_metrics)
df_dl_metrics.to_csv(OUT_DIR / "fold_metrics_deep_learning.csv", index=False)
df_dl_metrics


## 6. Deep Learning Results — Summary, ROC Curves, Confusion Matrices

In [ ]:
dl_summary = {}
for col in ["sensitivity_recall", "specificity", "precision", "f1", "auc"]:
    dl_summary[col] = {
        "mean": float(np.nanmean(df_dl_metrics[col])),
        "std": float(np.nanstd(df_dl_metrics[col])),
    }

with open(OUT_DIR / "summary_metrics_deep_learning.json", "w") as f:
    json.dump(dl_summary, f, indent=2)

print("=== Deep learning: cross-validated summary (mean ± std across folds) ===")
for k, v in dl_summary.items():
    print(f"  {k}: {v['mean']:.3f} ± {v['std']:.3f}")


In [ ]:
plt.figure(figsize=(6, 6))
for fold_idx, fpr, tpr, auc in dl_roc_data:
    plt.plot(fpr, tpr, label=f"Fold {fold_idx} (AUC={auc:.2f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves per fold — Deep Learning")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "roc_curves_deep_learning.png", dpi=150)
plt.show()


In [ ]:
n_cols = min(len(dl_cm_data), 5)
n_rows = int(np.ceil(len(dl_cm_data) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
axes = np.array(axes).reshape(-1)
for ax, (fold_idx, y_true, y_pred) in zip(axes, dl_cm_data):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    ax.imshow(cm, cmap="Blues")
    for (i, j), val in np.ndenumerate(cm):
        ax.text(j, i, str(val), ha="center", va="center")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["neg", "pos"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["neg", "pos"])
    ax.set_title(f"Fold {fold_idx}")
for ax in axes[len(dl_cm_data):]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrices_deep_learning.png", dpi=150)
plt.show()


## 7. Classical Machine Learning Baseline

Colour histogram + GLCM texture features, with SVM and Random Forest, using
the **same stratified k-fold protocol** for a fair comparison against the CNN.


In [ ]:
from skimage.feature import graycomatrix, graycoprops
from skimage.color import rgb2gray

def extract_features(img_path, size=(128, 128)):
    img = Image.open(img_path).convert("RGB").resize(size)
    arr = np.array(img)

    hist_features = []
    for c in range(3):
        hist, _ = np.histogram(arr[:, :, c], bins=16, range=(0, 255), density=True)
        hist_features.extend(hist)

    color_stats = []
    for c in range(3):
        channel = arr[:, :, c].astype(np.float32)
        color_stats.extend([channel.mean(), channel.std()])

    gray = (rgb2gray(arr) * 255).astype(np.uint8)
    glcm = graycomatrix(gray, distances=[1, 3], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                         levels=256, symmetric=True, normed=True)
    texture_features = []
    for prop in ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]:
        texture_features.extend(graycoprops(glcm, prop).flatten())

    return np.array(hist_features + color_stats + texture_features)


def compute_metrics_classical(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    specificity = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
    precision = precision_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = float("nan")
    return {"sensitivity_recall": sensitivity, "specificity": specificity,
            "precision": precision, "f1": f1, "auc": auc}

print("Classical feature extraction functions ready.")


In [ ]:
# Extract features for every image once (re-used across folds)
paths = [Path(p) for p in df_images["path"]]
y_all = df_images["label"].to_numpy()

print("Extracting handcrafted features for all images...")
X_all = np.array([extract_features(p) for p in paths])
print(f"Feature matrix shape: {X_all.shape}")


In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS if N_FOLDS != -1 else 5, shuffle=True, random_state=SEED)

classical_results = {"svm": [], "random_forest": []}

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_all, y_all), start=1):
    X_train, X_val = X_all[train_idx], X_all[val_idx]
    y_train, y_val = y_all[train_idx], y_all[val_idx]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)

    svm = SVC(kernel="rbf", probability=True, class_weight="balanced", C=1.0, random_state=SEED)
    svm.fit(X_train_s, y_train)
    svm_prob = svm.predict_proba(X_val_s)[:, 1]
    svm_pred = (svm_prob >= 0.5).astype(int)
    classical_results["svm"].append({**compute_metrics_classical(y_val, svm_pred, svm_prob), "fold": fold_idx})

    rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=SEED)
    rf.fit(X_train, y_train)
    rf_prob = rf.predict_proba(X_val)[:, 1]
    rf_pred = (rf_prob >= 0.5).astype(int)
    classical_results["random_forest"].append({**compute_metrics_classical(y_val, rf_pred, rf_prob), "fold": fold_idx})

    print(f"Fold {fold_idx}: SVM AUC={classical_results['svm'][-1]['auc']:.3f}  "
          f"RF AUC={classical_results['random_forest'][-1]['auc']:.3f}")

classical_summary = {}
for model_name, fold_results in classical_results.items():
    df = pd.DataFrame(fold_results)
    df.to_csv(OUT_DIR / f"baseline_{model_name}_fold_metrics.csv", index=False)
    classical_summary[model_name] = {
        col: {"mean": float(np.nanmean(df[col])), "std": float(np.nanstd(df[col]))}
        for col in ["sensitivity_recall", "specificity", "precision", "f1", "auc"]
    }

with open(OUT_DIR / "baseline_summary_metrics.json", "w") as f:
    json.dump(classical_summary, f, indent=2)

print("\n=== Classical baseline summary (mean ± std across folds) ===")
for model_name, metrics in classical_summary.items():
    print(f"\n{model_name}:")
    for k, v in metrics.items():
        print(f"  {k}: {v['mean']:.3f} ± {v['std']:.3f}")


## 8. Deep Learning vs. Classical Baseline — Side-by-Side Comparison

In [ ]:
comparison_rows = []
for metric in ["sensitivity_recall", "specificity", "precision", "f1", "auc"]:
    row = {"metric": metric}
    row["deep_learning (mean±std)"] = f"{dl_summary[metric]['mean']:.3f} ± {dl_summary[metric]['std']:.3f}"
    for model_name in ["svm", "random_forest"]:
        row[f"{model_name} (mean±std)"] = (
            f"{classical_summary[model_name][metric]['mean']:.3f} "
            f"± {classical_summary[model_name][metric]['std']:.3f}"
        )
    comparison_rows.append(row)

df_comparison = pd.DataFrame(comparison_rows)
df_comparison.to_csv(OUT_DIR / "model_comparison_summary.csv", index=False)
df_comparison


In [ ]:
# Bar chart comparing AUC across all models
model_labels = ["Deep Learning\n(" + BACKBONE + ")", "SVM", "Random Forest"]
auc_means = [dl_summary["auc"]["mean"], classical_summary["svm"]["auc"]["mean"],
             classical_summary["random_forest"]["auc"]["mean"]]
auc_stds = [dl_summary["auc"]["std"], classical_summary["svm"]["auc"]["std"],
            classical_summary["random_forest"]["auc"]["std"]]

plt.figure(figsize=(6, 4))
plt.bar(model_labels, auc_means, yerr=auc_stds, capsize=5,
        color=["#4C72B0", "#DD8452", "#55A868"])
plt.ylabel("AUC-ROC (mean ± SD across folds)")
plt.ylim(0, 1)
plt.title("Model comparison — AUC-ROC")
plt.tight_layout()
plt.savefig(OUT_DIR / "model_comparison_auc.png", dpi=150)
plt.show()


## 9. Grad-CAM — Model Interpretability

Generates a class-activation heatmap for a chosen image, showing which region
of the liver the model is basing its prediction on. Run this on a few example
images from each class using the best-performing fold's saved model.


In [ ]:
def build_model_for_gradcam(backbone_name):
    if backbone_name == "mobilenet_v2":
        model = models.mobilenet_v2(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)
        target_layer = model.features[-1]
    elif backbone_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)
        target_layer = model.features[-1]
    elif backbone_name == "resnet18":
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, 1)
        target_layer = model.layer4[-1]
    else:
        raise ValueError(f"Unknown backbone {backbone_name}")
    return model, target_layer


def gradcam(model, target_layer, input_tensor):
    activations, gradients = {}, {}

    def fwd_hook(module, inp, out):
        activations["value"] = out

    def bwd_hook(module, grad_in, grad_out):
        gradients["value"] = grad_out[0]

    h1 = target_layer.register_forward_hook(fwd_hook)
    h2 = target_layer.register_full_backward_hook(bwd_hook)

    model.zero_grad()
    output = model(input_tensor)
    output[:, 0].backward()

    h1.remove(); h2.remove()

    acts = activations["value"][0]
    grads = gradients["value"][0]
    weights = grads.mean(dim=(1, 2))
    cam = torch.relu((weights[:, None, None] * acts).sum(dim=0))
    cam = cam / (cam.max() + 1e-8)
    return cam.detach().cpu().numpy(), torch.sigmoid(output).item()


def show_gradcam(image_path, model_path, backbone_name=BACKBONE, image_size=IMAGE_SIZE):
    import matplotlib.cm as _cm
    model, target_layer = build_model_for_gradcam(backbone_name)
    model.load_state_dict(torch.load(model_path, map_location="cpu"))
    model.eval()

    tf = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    orig_img = Image.open(image_path).convert("RGB").resize((image_size, image_size))
    input_tensor = tf(orig_img).unsqueeze(0)
    input_tensor.requires_grad_(True)

    cam, prob = gradcam(model, target_layer, input_tensor)
    cam_resized = np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize(
        (image_size, image_size), Image.BILINEAR)) / 255.0
    heatmap = _cm.jet(cam_resized)[:, :, :3]
    overlay = 0.5 * np.array(orig_img) / 255.0 + 0.5 * heatmap

    fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
    axes[0].imshow(orig_img); axes[0].set_title("Original"); axes[0].axis("off")
    axes[1].imshow(cam_resized, cmap="jet"); axes[1].set_title("Grad-CAM"); axes[1].axis("off")
    axes[2].imshow(overlay); axes[2].set_title(f"Overlay (P(positive)={prob:.2f})"); axes[2].axis("off")
    plt.tight_layout()
    plt.show()
    return prob


In [ ]:
# Example: run Grad-CAM on one positive and one negative example, using fold 1's model
example_pos = df_images[df_images["class"] == "positive"]["path"].iloc[0]
example_neg = df_images[df_images["class"] == "negative"]["path"].iloc[0]
model_path = OUT_DIR / "best_model_fold1.pt"

if model_path.exists():
    print("Positive example:")
    show_gradcam(example_pos, model_path)
    print("Negative example:")
    show_gradcam(example_neg, model_path)
else:
    print(f"{model_path} not found — run the training section (Section 5) first.")


## 10. Summary & Next Steps

- Cross-validated results for both the deep learning model and the classical
  baseline are saved in `outputs/` as CSV/JSON, alongside ROC curves,
  confusion matrices, and a side-by-side comparison table/chart.
- Report the **mean ± SD across folds**, not a single fold's number, given the
  small per-fold validation set size at n=100.
- Treat these results as **pilot/feasibility estimates** in any write-up.
- As more images are collected (target ≥300–500 per class), re-run this same
  notebook unchanged — the pipeline scales directly to a larger dataset.
- Once a larger dataset is available, hold out an **entirely separate batch**
  (different day/animals, ideally different site) as an external test set —
  this is the strongest test of real-world generalisation.
- If/when lesion sub-type labels (fascioliasis, abscess, fibrosis, cirrhosis,
  fatty liver) are available in sufficient numbers per category, this
  pipeline can be extended from binary to multi-class classification by
  changing the final layer to `num_classes` outputs and switching to
  `CrossEntropyLoss` with softmax.
